# Python OOP

## `README.md` pro studenty

```markdown
# OOP cvičení – Python

## Struktura
- `cviceni_XX_*.py` – šablony k doplnění
- `testy/test_XX.py` – testy (spouštěj průběžně)

## Jak začít
1. Otevři šablonu, přečti docstring a `# TODO`.
2. Doplň logiku.
3. Spusť `python cviceni_XX_*.py` – zkontroluj `__main__` výstup.
4. Spusť `pytest testy/test_XX.py -v`.
5. Když vše zelené, jdi na další cvičení.

## Instalace
    pip install pytest

## Doporučené pořadí
1 → 2 → 3 → 4 → 5 → 7 → 6 → 8 → 10 → 11 → 9 → 12
```

## Metodické tipy

| Krok | Co student dělá | Co učitel sleduje |
|---|---|---|
| **Čtení šablony** | Pochopí docstring a typy | Správně interpretuje zadání? |
| **TODO** | Doplní jen prázdná místa | Nerozbíjí existující kostru |
| **Spuštění `__main__`** | Vidí okamžitou zpětnou vazbu | Rozumí výstupům? |
| **pytest** | Ověří hraniční případy | Chápe, proč test selhal? |
| **Reflexe** | Vysvětlí, proč použil property/dědičnost | Umí obhájit návrhové rozhodnutí |

**Hodnocení:** 50 % pytest zelený,

# OOP v Pythonu – praktické příklady pro výuku

Níže najdeš **12 řešených praktických příkladů** zaměřených čistě na OOP, seřazených od základů po pokročilé. Každý má zadání, řešení s komentáři a vysvětlení konceptu.

---

## 1. Základy třídy a instance

### Příklad 1: Třída `BankovniUcet`

**Zadání:** Vytvoř třídu s vlastnostmi `majitel` a `zustatek`. Přidej metody `vloz()` a `vyber()` s ochranou proti přečerpání.


In [15]:
class BankovniUcet:
    def __init__(self, majitel, zustatek=0):
        self.majitel = majitel
        self.zustatek = zustatek

    def vloz(self, castka):
        if castka <= 0:
            raise ValueError("Vklad musí být kladný")
        self.zustatek += castka
        return self.zustatek

    def vyber(self, castka):
        if castka <= 0:
            raise ValueError("Výběr musí být kladný")
        if castka > self.zustatek:
            raise ValueError(f"Nedostatek prostředků (máš {self.zustatek} Kč)")
        self.zustatek -= castka
        return self.zustatek

    def __str__(self):
        return f"Účet {self.majitel}: {self.zustatek} Kč"


In [16]:
ucet = BankovniUcet("Anna", 1000)
ucet.vloz(500)
print(ucet)              # Účet Anna: 1500 Kč
ucet.vyber(2000)         # ValueError: Nedostatek prostředků

Účet Anna: 1500 Kč


ValueError: Nedostatek prostředků (máš 1500 Kč)

**Vysvětlení:** `__init__` inicializuje instanci. `self` odkazuje na konkrétní objekt. Validace patří do metod – chrání stav objektu. `__str__` definuje čitelný výpis pro `print()`.


## 2. Zapouzdření a `@property`

### Příklad 2: Validovaný věk přes property

**Zadání:** U třídy `Osoba` zajisti, že věk nelze nastavit záporný – použij `@property`.


In [17]:
class Osoba:
    def __init__(self, jmeno, vek):
        self.jmeno = jmeno
        self.vek = vek          # volá setter

    @property
    def vek(self):
        return self._vek

    @vek.setter
    def vek(self, hodnota):
        if not isinstance(hodnota, int):
            raise TypeError("Věk musí být celé číslo")
        if hodnota < 0 or hodnota > 150:
            raise ValueError("Neplatný věk")
        self._vek = hodnota

    @property
    def plnolety(self):          # read-only odvozená vlastnost
        return self._vek >= 18


In [18]:

o = Osoba("Petr", 25)
print(o.plnolety)           # True
o.vek = 30                  # OK
o.vek = -5                  # ValueError: Neplatný věk

True


ValueError: Neplatný věk

**Vysvětlení:** `@property` zpřístupní metodu jako atribut. Setter validuje přiřazení. Podtržítko `_vek` značí "interní" atribut. Property bez setteru = read-only (`plnolety`).


## 3. Dědičnost a `super()`

### Příklad 3: Hierarchie zvířat

**Zadání:** Vytvoř základní třídu `Zvire` a potomky `Pes`, `Kocka`, kteří přepisují metodu `zvuk()`. Použij `super()` v konstruktoru.


In [6]:
class Zvire:
    def __init__(self, jmeno, vek):
        self.jmeno = jmeno
        self.vek = vek

    def zvuk(self):
        raise NotImplementedError("Potomek musí implementovat zvuk()")

    def info(self):
        return f"{type(self).__name__} {self.jmeno} ({self.vek} let): {self.zvuk()}"

class Pes(Zvire):
    def __init__(self, jmeno, vek, plemeno):
        super().__init__(jmeno, vek)     # zavolá konstruktor rodiče
        self.plemeno = plemeno

    def zvuk(self):
        return "Haf!"

class Kocka(Zvire):
    def zvuk(self):
        return "Mňau!"

In [ ]:
for z in [Pes("Rex", 3, "labrador"), Kocka("Micka", 2)]:
    print(z.info())
# Pes Rex (3 let): Haf!
# Kocka Micka (2 let): Mňau!


**Vysvětlení:** `super().__init__()` zavolá konstruktor rodiče – nemusíš opakovat kód. `NotImplementedError` vynutí implementaci v potomcích. Polymorfismus: `z.zvuk()` se rozhodne až za běhu podle typu.


## 4. Polymorfismus přes společný interface

### Příklad 4: Tvary a výpočet obsahu

**Zadání:** Vytvoř třídy `Ctverec`, `Obdelnik`, `Kruh` se společnou metodou `obsah()` a funkcí, která spočítá celkový obsah seznamu tvarů.


In [19]:
import math

class Tvar:
    def obsah(self):
        raise NotImplementedError

    def __str__(self):
        return f"{type(self).__name__}: obsah={self.obsah():.2f}"

class Ctverec(Tvar):
    def __init__(self, a):
        self.a = a
    def obsah(self):
        return self.a ** 2

class Obdelnik(Tvar):
    def __init__(self, a, b):
        self.a, self.b = a, b
    def obsah(self):
        return self.a * self.b

class Kruh(Tvar):
    def __init__(self, r):
        self.r = r
    def obsah(self):
        return math.pi * self.r ** 2


In [20]:
tvary = [Ctverec(3), Obdelnik(2, 5), Kruh(4)]
for t in tvary:
    print(t)

celkem = sum(t.obsah() for t in tvary)
print(f"Celkem: {celkem:.2f}")

Ctverec: obsah=9.00
Obdelnik: obsah=10.00
Kruh: obsah=50.27
Celkem: 69.27


**Vysvětlení:** Všechny tvary sdílejí metodu `obsah()` – volající kód nemusí znát konkrétní typ. `sum()` s generátorem funguje díky polymorfismu. Základ pro návrhový vzor *Strategy*.


## 5. Třídní atributy a metody

### Příklad 5: Počítadlo instancí

**Zadání:** Vytvoř třídu `Uzivatel`, která si pamatuje celkový počet vytvořených uživatelů. Přidej třídní metodu pro výpis.


In [21]:
class Uzivatel:
    pocet = 0                          # třídní atribut (sdílený)

    def __init__(self, jmeno):
        self.jmeno = jmeno
        Uzivatel.pocet += 1

    @classmethod
    def celkem(cls):                   # třídní metoda – pracuje s cls
        return cls.pocet

    @staticmethod
    def validuj_jmeno(jmeno):          # statická metoda – nepotřebuje cls ani self
        return isinstance(jmeno, str) and len(jmeno) >= 2

    def __del__(self):                 # destruktor
        Uzivatel.pocet -= 1


In [22]:
a = Uzivatel("Anna")
b = Uzivatel("Petr")
print(Uzivatel.celkem())             # 2
print(Uzivatel.validuj_jmeno("Al"))  # True
del a
print(Uzivatel.celkem())             # 1

2
True
1


In [25]:
c = Uzivatel("Marta")
b = Uzivatel("Petr")
print(Uzivatel.celkem())  

3


**Vysvětlení:**
- **Třídní atribut** (`pocet`) – společný všem instancím.
- **`@classmethod`** – dostane třídu `cls`, typicky tovární metody nebo práce se stavem třídy.
- **`@staticmethod`** – pomocná funkce logicky patřící ke třídě, ale nepotřebuje instanci.
- **`__del__`** – destruktor (v praxi se mu spíš vyhýbej, použij context manager).


## 6. Dunder metody – operátory

### Příklad 6: Třída `Zlomek` s operátory

**Zadání:** Implementuj zlomek, který podporuje `+`, `-`, `*`, `/`, `==` a automatické krácení.


In [ ]:
import math

class Zlomek:
    def __init__(self, citatel, jmenovatel):
        if jmenovatel == 0:
            raise ZeroDivisionError("Jmenovatel nesmí být 0")
        if jmenovatel < 0:              # znaménko vždy v čitateli
            citatel, jmenovatel = -citatel, -jmenovatel
        d = math.gcd(citatel, jmenovatel)
        self.citatel = citatel // d
        self.jmenovatel = jmenovatel // d

    def __add__(self, other):
        return Zlomek(
            self.citatel * other.jmenovatel + other.citatel * self.jmenovatel,
            self.jmenovatel * other.jmenovatel,
        )

    def __sub__(self, other):
        return Zlomek(
            self.citatel * other.jmenovatel - other.citatel * self.jmenovatel,
            self.jmenovatel * other.jmenovatel,
        )

    def __mul__(self, other):
        return Zlomek(self.citatel * other.citatel,
                      self.jmenovatel * other.jmenovatel)

    def __truediv__(self, other):
        return Zlomek(self.citatel * other.jmenovatel,
                      self.jmenovatel * other.citatel)

    def __eq__(self, other):
        return (isinstance(other, Zlomek)
                and self.citatel == other.citatel
                and self.jmenovatel == other.jmenovatel)

    def __repr__(self):
        return f"Zlomek({self.citatel}, {self.jmenovatel})"

    def __str__(self):
        return f"{self.citatel}/{self.jmenovatel}"


In [ ]:
a = Zlomek(1, 2)
b = Zlomek(1, 3)
print(a + b)     # 5/6
print(a - b)     # 1/6
print(a * b)     # 1/6
print(a / b)     # 3/2
print(a == Zlomek(2, 4))   # True (automaticky zkráceno)

**Vysvětlení:** Dunder metody umožní přirozenou syntaxi (`a + b`). Krácení v `__init__` zaručí kanonický tvar – proto `1/2 == 2/4`. `__repr__` = vývojářský výpis, `__str__` = uživatelský.


## 7. Kompozice vs. dědičnost

### Příklad 7: Auto a motor (kompozice)

**Zadání:** Místo dědění `Auto(Motor)` použij kompozici – auto *má* motor.


In [ ]:
class Motor:
    def __init__(self, vykon):
        self.vykon = vykon
        self.bezi = False

    def start(self):
        self.bezi = True
        return "Motor nastartován"

    def stop(self):
        self.bezi = False
        return "Motor vypnut"

class Auto:
    def __init__(self, znacka, motor):
        self.znacka = znacka
        self.motor = motor           # HAS-A vztah (kompozice)

    def nastartuj(self):
        if self.motor.bezi:
            return "Už běží"
        return self.motor.start()

    def __str__(self):
        return f"{self.znacka} ({self.motor.vykon} kW)"


In [ ]:
auto = Auto("Škoda", Motor(110))
print(auto)                  # Škoda (110 kW)
print(auto.nastartuj())      # Motor nastartován
print(auto.nastartuj())      # Už běží

In [ ]:
**Vysvětlení:** Dědičnost = *je* (Pes je Zvíře). Kompozice = *má* (Auto má Motor). Kompozice je flexibilnější – motor můžeš vyměnit, snadno testovat, vyhnout se křehké hierarchii. **Preferuj kompozici před dědičností.**


## 8. Speciální metody – kontejner

### Příklad 8: Třída `Kosik` jako kontejner

**Zadání:** Vytvoř nákupní košík, který podporuje `len()`, `in`, iteraci a indexování.


In [8]:
class Kosik:
    def __init__(self):
        self._polozky = []

    def pridej(self, nazev, cena):
        self._polozky.append({"nazev": nazev, "cena": cena})

    def __len__(self):
        return len(self._polozky)

    def __getitem__(self, index):
        return self._polozky[index]

    def __iter__(self):
        return iter(self._polozky)

    def __contains__(self, nazev):
        return any(p["nazev"] == nazev for p in self._polozky)

    def celkova_cena(self):
        return sum(p["cena"] for p in self._polozky)

In [ ]:
k = Kosik()
k.pridej("Chleba", 45)
k.pridej("Mléko", 25)
k.pridej("Sýr", 120)

print(len(k))               # 3
print("Mléko" in k)         # True
print(k[0])                 # {'nazev': 'Chleba', 'cena': 45}
print(k.celkova_cena())     # 190

for p in k:                 # iterace
    print(p["nazev"])

**Vysvětlení:** `__len__`, `__getitem__`, `__iter__`, `__contains__` udělají z objektu "kolekci" – funguje s `len()`, `in`, `for`, indexováním. Duck typing: chová se jako seznam.


## 9. Dědičnost od vestavěných typů

### Příklad 9: Vlastní seznam s validací

**Zadání:** Vytvoř `KladnySeznam`, který povolí jen kladná čísla, ale jinak se chová jako běžný seznam.


In [ ]:
class KladnySeznam(list):
    def _zkontroluj(self, hodnota):
        if not isinstance(hodnota, (int, float)) or hodnota <= 0:
            raise ValueError(f"Jen kladná čísla, dostal jsem {hodnota!r}")

    def append(self, hodnota):
        self._zkontroluj(hodnota)
        super().append(hodnota)

    def extend(self, hodnoty):
        for h in hodnoty:
            self._zkontroluj(h)
        super().extend(hodnoty)

    def insert(self, index, hodnota):
        self._zkontroluj(hodnota)
        super().insert(index, hodnota)


In [ ]:
s = KladnySeznam([1, 2, 3])
s.append(4)
print(s)                    # [1, 2, 3, 4]
s.append(-1)                # ValueError
s.extend([5, -2])           # ValueError (atomicky – nic se nepřidá)

**Vysvětlení:** Dědičnost od `list` je pohodlná, ale pozor – přepisuješ jen některé metody (`+=` volá `__iadd__`, ne `extend`). Pro plnou kontrolu bys musel pokrýt víc metod. Alternativa: kompozice s interním `list`.


In [ ]:
Ve třídě KladnySeznam se nepíše __init__ ani super().__init__(), protože třída nepotřebuje při svém vzniku dělat 
nic navíc oproti běžnému seznamu (list).

Proč tam __init__ chybí?

• Automatické dědění: Když v potomkovi nedefinujete vlastní __init__, Python automaticky použije __init__ z rodičovské třídy list.
• Žádné nové atributy: KladnySeznam si nepotřebuje pamatovat žádné vlastní proměnné (jako např. self.nazev), stačí mu jen samotná data v seznamu, 
o která se postará list.
• Bez zbytečného kódu: Psát prázdný konstruktor, který jen zavolá rodiče, je v Pythonu zbytečné:python
# Toto by dělalo úplně to samé, jako když tam __init__ vůbec není:
def __init__(self, *args, **kwargs):
    super().__init__(*args, **kwargs)
Use code with caution.

Pozor na skrytou chybu v tomto kódu!

Protože tam vlastní __init__ není, kontrola _zkontroluj() probíhá jen při volání append(), extend() a insert(). 
    Pokud seznam vytvoříte rovnou s hodnotami na začátku, kontrola se obejde:
python
s = KladnySeznam([1, -5, "ahoj"])  # Projde bez chyby! Zavolal se přímo list.__init__
s.append(-5)                       # Tady už to vyhodí ValueError
Use code with caution.

Kdy byste tam __init__ a super().__init__() musela přidat?

Právě tehdy, kdybyste chtěla hlídat i počáteční data při vytváření seznamu:
python
class KladnySeznam(list):
    def __init__(self, polozky=()):
        super().__init__()      # Vytvoří prázdný list
        self.extend(polozky)    # Využije vaši upravenou metodu extend (a vše zkontroluje!)
Use code with caution.

Shrnutí

• super().__init__() píšete jen tehdy, když v potomkovi potřebujete změnit nebo rozšířit chování při vytváření objektu (např. přidat nový atribut nebo zkontrolovat vstupní data).
• super().append(), super().extend() a super().insert() v kódu máte proto, že právě u těchto tří metod jste chování oproti běžnému list změnila (přidala jste kontrolu _zkontroluj).


In [ ]:
class KladnySeznam(list):
    def __init__(self, polozky=()):
        super().__init__()          # 1. Vytvoří prázdný seznam
        self.extend(polozky)        # 2. Přidá prvky přes vaši metodu extend (s kontrolou)

    def _zkontroluj(self, hodnota):
        # Poznámka: isinstance(True, int) je v Pythonu True, proto je dobré vyloučit i bool
        if isinstance(hodnota, bool) or not isinstance(hodnota, (int, float)) or hodnota <= 0:
            raise ValueError(f"Jen kladná čísla, dostal jsem {hodnota!r}")

    def append(self, hodnota):
        self._zkontroluj(hodnota)
        super().append(hodnota)

    def extend(self, hodnoty):
        for h in hodnoty:
            self._zkontroluj(h)
        super().extend(hodnoty)

    def insert(self, index, hodnota):
        self._zkontroluj(hodnota)
        super().insert(index, hodnota)


## 10. `@dataclass` a `@property` v praxi

### Příklad 10: Produkt s vypočítanou cenou

**Zadání:** Vytvoř datovou třídu `Produkt` s cenou bez DPH, sazbou DPH a odvozenou cenou s DPH. Přidej metodu pro slevu.


In [9]:
from dataclasses import dataclass

@dataclass
class Produkt:
    nazev: str
    cena_bez_dph: float
    sazba_dph: float = 21.0

    def __post_init__(self):
        if self.cena_bez_dph < 0:
            raise ValueError("Cena nesmí být záporná")
        if self.sazba_dph not in (0, 12, 21):
            raise ValueError("Neplatná sazba DPH")

    @property
    def cena_s_dph(self):
        return round(self.cena_bez_dph * (1 + self.sazba_dph / 100), 2)

    def sleva(self, procenta):
        if not 0 <= procenta <= 100:
            raise ValueError("Sleva musí být 0–100 %")
        self.cena_bez_dph = round(self.cena_bez_dph * (1 - procenta / 100), 2)


In [ ]:
p = Produkt("Notebook", 20000)
print(p)                    # Produkt(nazev='Notebook', cena_bez_dph=20000, sazba_dph=21.0)
print(p.cena_s_dph)         # 24200.0
p.sleva(10)
print(p.cena_s_dph)         # 21780.0


**Vysvětlení:** `@dataclass` vygeneruje `__init__`, `__repr__`, `__eq__`. `__post_init__` slouží pro validaci. `@property` přidá odvozený atribut, který se počítá dynamicky – vždy aktuální.


# Property

In [ ]:
class Property:
    def __init__(self, square_feet='', beds='',
            baths='', **kwargs):
        super().__init__(**kwargs)
        self.square_feet = square_feet
        self.num_bedrooms = beds
        self.num_baths = baths

    def display(self):
        print("PROPERTY DETAILS")
        print("================")
        print("square footage: {}".format(self.square_feet))
        print("bedrooms: {}".format(self.num_bedrooms))
        print("bathrooms: {}".format(self.num_baths))
        print()

    def prompt_init():
        return dict(square_feet=input("Enter the square feet: "),
                beds=input("Enter number of bedrooms: "),
                baths=input("Enter number of baths: "))
    prompt_init = staticmethod(prompt_init)

In [ ]:
x=Property(15,22,3,"Mám pořád málo")

In [ ]:
Váš kód selže ze dvou důvodů: posíláte poziční argument místo pojmenovaného a zároveň posíláte argument, se kterým si rodičovská třída object neumí poradit.

1. Poziční vs. pojmenovaný argument (*args vs. **kwargs)

• **kwargs (s dvěma hvězdičkami) přijímá pouze pojmenované argumenty ve tvaru klic=hodnota (např. poznamka="Mám pořád málo").
• Vy jste napsali "Mám pořád málo" jako 4. poziční argument bez názvu. Python očekává jen 3 poziční argumenty (square_feet, beds, baths), proto okamžitě vyhodí chybu:
TypeError: Property.__init__() takes from 1 to 4 positional arguments but 5 were given

2. Proč by to spadlo i s pojmenovaným argumentem?

I kdybyste volání opravili na:
python
x = Property(15, 22, 3, poznamka="Mám pořád málo")
Use code with caution.
Program spadne na řádku super().__init__(**kwargs) s chybou:
TypeError: object.__init__() takes exactly one argument (the instance to initialize)
Důvod: Třída Property nemá uvedeného žádného rodiče, takže v Pythonu automaticky dědí z základní třídy object. Konstruktor třídy object.__init__() žádné extra argumenty nepřijímá – pokud do něj přes **kwargs něco pošlete, vyhodí chybu.

K čemu tam tedy **kwargs a super().__init__(**kwargs) jsou?

Tento kód je klasický příklad vícenásobné dědičnosti (z knihy Python 3 Object-Oriented Programming). Předpokládá se, že vznikne další třída (např. House nebo Apartment), která zdědí vlastnosti z Property a zároveň z jiné třídy (třeba Purchase nebo Rental):
python
class Purchase:
    def __init__(self, price='', taxes='', **kwargs):
        super().__init__(**kwargs)
        self.price = price
        self.taxes = taxes

# Třída dědí z obou tříd najednou
class HousePurchase(Property, Purchase):
    pass

# Tady to projde! 'price' a 'taxes' se zachytí do **kwargs v Property,
# pošlou se přes super() do Purchase, a do object.__init__() už dojde prázdný slovník {}
x = HousePurchase(square_feet=15, beds=2, baths=3, price=5000000, taxes=10000)
Use code with caution.
• Property si vezme své parametry (square_feet, beds, baths) a zbytek (price, taxes) pošle dál přes super().__init__(**kwargs).
• Třída Purchase si z nich vybere price a taxes, takže **kwargs se úplně vyprázdní.
• Nakonec se zavolá object.__init__() s prázdnými **kwargs (nic se nepředá) a vše proběhne bez chyby.
Chcete upravit třídu Property tak, aby uměla ukládat libovolné vlastní poznámky z **kwargs přímo do self, nebo pokračujete v příkladu s vícenásobnou dědičností (třídy Apartment, House, Rental)?


In [ ]:
Abyste mohla **kwargs použít pro přidání libovolných nových argumentů přímo do třídy Property, musíte udělat dvě změny:
1. Zrušit posílání **kwargs do super().__init__() (protože rodičovská třída object žádné extra argumenty nepřijímá a shodí program).
2. Sama ty argumenty uložit do objektu self (např. jako slovník nebo jako nové atributy).

Opravený kód (2 způsoby, jak to udělat)


Způsob A: Uložit je jako slovník (nejbezpečnější)

python
class Property:
    def __init__(self, square_feet='', beds='', baths='', **kwargs):
        super().__init__()           # <-- Voláme bez **kwargs!
        self.square_feet = square_feet
        self.num_bedrooms = beds
        self.num_baths = baths
        self.dalsi_info = kwargs     # <-- Tady si uložíme zbytek

# Volání MUSÍ mít název parametru (klíč=hodnota):
x = Property(15, 22, 3, poznamka="Mám pořád málo", balkon=True)

print(x.dalsi_info)  # {'poznamka': 'Mám pořád málo', 'balkon': True}
Use code with caution.

Způsob B: Vytvořit z nich přímo atributy objektu (x.poznamka)

Pokud chcete k novým údajům přistupovat přes tečku (stejně jako k x.square_feet), použijte smyčku a funkci setattr():
python
class Property:
    def __init__(self, square_feet='', beds='', baths='', **kwargs):
        super().__init__()           # <-- Voláme bez **kwargs!
        self.square_feet = square_feet
        self.num_bedrooms = beds
        self.num_baths = baths
        
        # Projde všechny extra argumenty a vytvoří z nich atributy
        for klic, hodnota in kwargs.items():
            setattr(self, klic, hodnota)

# Volání:
x = Property(15, 22, 3, poznamka="Mám pořád málo", patro=4)

print(x.poznamka)  # Mám pořád málo
print(x.patro)     # 4
Use code with caution.

Proč to předtím nešlo? Shrnutí ve 3 bodech

• **kwargs bere jen nazev=hodnota: Nemůžete napsat jen "Mám pořád málo". Musíte napsat poznamka="Mám pořád málo". (Kdybyste chtěla posílat hodnoty bez názvu, musela byste v def __init__ použít *args s jednou hvězdičkou).
• super().__init__(**kwargs) je "posílání dál": V knize se **kwargs používá jen jako pošťák, který předává parametry dalším třídám v řadě.
• Konec řady je object: Když Property nemá dalšího rodiče, na konci řady stojí základní třída object, která jakékoliv extra argumenty odmítne chybou. Proto je musíte "spotřebovat" už v Property a dál poslat prázdné super().__init__().
Chcete také ukázat, jak upravit metodu display(), aby tyto nově přidané argumenty z **kwargs automaticky vypsala, nebo jak by to vypadalo s použitím *args (bez názvů parametrů)?


In [ ]:
class Property:
    def __init__(self, square_feet='', beds='', baths='', **kwargs):
        super().__init__()           # <-- Voláme bez **kwargs!
        self.square_feet = square_feet
        self.num_bedrooms = beds
        self.num_baths = baths
        self.dalsi_info = kwargs     # <-- Tady si uložíme zbytek

# Volání MUSÍ mít název parametru (klíč=hodnota):
x = Property(15, 22, 3, poznamka="Mám pořád málo", balkon=True)

## 11. Abstraktní třídy

### Příklad 11: Abstraktní `Platba` s implementacemi

**Zadání:** Pomocí `abc` vytvoř abstraktní třídu `Platba` s metodou `proved()`, a konkrétní `Kartou`, `Hotove`, `Prevodem`.


In [26]:
from abc import ABC, abstractmethod

class Platba(ABC):
    def __init__(self, castka):
        if castka <= 0:
            raise ValueError("Částka musí být kladná")
        self.castka = castka

    @abstractmethod
    def proved(self):
        """Provede platbu a vrátí potvrzení."""
        ...

    def __str__(self):
        return f"{type(self).__name__}: {self.castka} Kč"

class Kartou(Platba):
    def __init__(self, castka, cislo_karty):
        super().__init__(castka)
        self.cislo_karty = cislo_karty
    def proved(self):
        return f"Zaplaceno kartou ****{self.cislo_karty[-4:]}"

class Hotove(Platba):
    def proved(self):
        return f"Přijato {self.castka} Kč v hotovosti"

class Prevodem(Platba):
    def __init__(self, castka, ucet):
        super().__init__(castka)
        self.ucet = ucet
    def proved(self):
        return f"Převedeno na účet {self.ucet}"


In [27]:
platby = [Kartou(500, "1234567890123456"), Hotove(200), Prevodem(1000, "123-456")]
for p in platby:
    print(f"{p} → {p.proved()}")

# Platba(100)  # TypeError: Can't instantiate abstract class

Kartou: 500 Kč → Zaplaceno kartou ****3456
Hotove: 200 Kč → Přijato 200 Kč v hotovosti
Prevodem: 1000 Kč → Převedeno na účet 123-456


**Vysvětlení:** `ABC` + `@abstractmethod` vynutí implementaci – nelze vytvořit instanci abstraktní třídy. Skvělý způsob, jak definovat "smlouvu" pro potomky. V praxi základ pro pluginy, strategie, šablony.


## 12. Magic metody + návrhový vzor Singleton

### Příklad 12: Singleton přes `__new__` (konfigurace)

**Zadání:** Vytvoř třídu `Konfigurace`, která má vždy jen jednu instanci (Singleton) a umožňuje přístup jako ke slovníku.


In [ ]:
class Konfigurace:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._data = {}
        return cls._instance

    def __setitem__(self, klic, hodnota):
        self._data[klic] = hodnota

    def __getitem__(self, klic):
        return self._data[klic]

    def __contains__(self, klic):
        return klic in self._data

    def __repr__(self):
        return f"Konfigurace({self._data})"


In [ ]:
k1 = Konfigurace()
k1["host"] = "localhost"
k1["port"] = 8080

k2 = Konfigurace()          # stejná instance!
print(k1 is k2)             # True
print(k2["host"])           # localhost
print("port" in k2)         # True
k2["debug"] = True
print(k1)                   # Konfigurace({'host': 'localhost', 'port': 8080, 'debug': True})

**Vysvětlení:** `__new__` řídí **vytváření** instance (na rozdíl od `__init__`, který ji inicializuje). Singleton zajistí, že `Konfigurace()` vždy vrátí stejný objekt. `__setitem__`/`__getitem__` umožní přístup `k["klic"]` jako u dictu. *Pozor:* v produkci často lepší než Singleton je Dependency Injection – uveď to studentům jako diskuzi.


# Cvičení bez řešení (pro studenty)

### Cvičení 1 – Základy *(lehké)*
Vytvoř třídu `Heslo`, která v konstruktoru přijme řetězec. Přidej metodu `je_bezpecne()`, která vrátí `True`, pokud má alespoň 8 znaků, obsahuje číslici a velké písmeno. Přepiš `__str__` tak, aby se heslo nevypisovalo, ale jen "Heslo (skryto)".
> **Hint:** `any(c.isdigit() for c in ...)`, `__str__` vrací bezpečný text.


### Cvičení 2 – Property *(lehké)*
Třída `Teplota` s atributem `celsius` (validace −273,15 až 1000). Přidej read-only property `fahrenheit` (`C * 9/5 + 32`) a `kelvin` (`C + 273,15`).
> **Hint:** `@property` bez setteru pro převody.


### Cvičení 3 – Dědičnost *(střední)*
Vytvoř základní třídu `Zamestnanec` (jméno, plat) a potomky `Manazer` (příplatek) a `Programator` (bonus za projekt). Každý má metodu `celkova_odmena()`. Napiš funkci, která sečte odměny všech zaměstnanců v seznamu.
> **Hint:** `super().__init__()`, přepiš `celkova_odmena()`, polymorfní `sum()`.


### Cvičení 4 – Dunder metody *(střední)*
Implementuj třídu `Penize` s měnou a částkou. Podporuj `+`, `-`, `==` (jen stejná měna, jinak `ValueError`), `__str__` formátující jako `"1 250,00 Kč"`.
> **Hint:** formátování `f"{castka:,.2f}".replace(",", " ").replace(".", ",")`, kontrola měny v operátorech.


### Cvičení 5 – Kompozice *(střední)*
Vytvoř třídy `Autor` (jméno, rok narození) a `Kniha` (název, autor, rok). Kniha *má* autora. Přidej metodu `stari_pri_vydani()`. Vytvoř seznam knih a najdi nejstarší knihu podle roku vydání.
> **Hint:** `min(seznam, key=...)`, kompozice přes atribut `self.autor`.


### Cvičení 6 – Abstraktní třídy *(střední)*
Pomocí `ABC` vytvoř abstraktní `Zvire` s `@abstractmethod zvuk()` a `popis()`. Potomci `Pes`, `Kocka`, `Krava`. Přidej funkci, která vrátí seznam zvuků pro seznam zvířat.
> **Hint:** `ABC`, `@abstractmethod`, ověř, že `Zvire()` vyhodí `TypeError`.


### Cvičení 7 – Kontejner *(těžší)*
Vytvoř třídu `Fronta` (queue) s metodami `enqueue`, `dequeue`, `__len__`, `__bool__`, `__iter__`. Pokud je fronta prázdná, `dequeue()` vyhodí vlastní výjimku `PrazdnaFronta`.
> **Hint:** interní `collections.deque`, `__bool__` vrací `len(self) > 0`, vlastní výjimka dědí od `Exception`.


### Cvičení 8 – Třídní atributy *(těžší)*
Vytvoř třídu `Student` s automatickým `id` (inkrementální číslo), seznamem známek a metodou `prumer()`. Přidej třídní metodu `z_csv(text)`, která ze stringu typu `"Anna;1,2,3\nPetr;2,2"` vytvoří seznam studentů.
> **Hint:** `@classmethod`, `cls.pocitadlo += 1`, `text.strip().split("\n")`, pak `split(";")` a `split(",")`.


### Cvičení 9 – Dědičnost od vestavěného typu *(těžší)*
Vytvoř `KlicSlovnik(dict)`, který při přístupu na neexistující klíč vrátí `None` místo `KeyError`, a při přiřazení klíče typu `str` ho převede na lowercase. Ošetři i `get`, `setdefault` a `__contains__`.
> **Hint:** přepiš `__getitem__`, `__setitem__`, `__contains__`; pozor na nekonzistenci s `get`.


### Cvičení 10 – Návrhový vzor *(těžké)*
Implementuj jednoduchý **Observer** pattern: třída `Novinky` (subject) s metodami `prihlas(pozorovatel)` a `posli(zprava)`. Pozorovatelé jsou objekty s metodou `oznam(zprava)`. Otestuj se dvěma odběrateli.
> **Hint:** seznam pozorovatelů, v `posli()` je projdi a zavolej `oznam()`.


### Cvičení 11 – Kombinace *(těžké)*
Vytvoř systém pro e-shop: abstraktní `Produkt` (`cena()`, `popis()`), konkrétní `Kniha`, `Elektronika`, `Potravina` (s vlastními pravidly DPH). Třída `Kosik` (kontejner s `__len__`, `__iter__`, `celkem()`) a `Objednavka`, která umí aplikovat slevový kupón. Vše otestuj alespoň 3 asserty.
> **Hint:** kombinace příkladů 4, 6, 8, 11; `@property` pro odvozené ceny.


### Cvičení 12 – Refaktoring *(těžké)*
Následující "procedurální" kód přepiš do OOP s třídami `Ucet`, `Transakce`, `Banka`. Přidej validace a metodu `historie()`:

```python
ucty = {"Anna": 1000, "Petr": 500}
ucty["Anna"] -= 200
ucty["Petr"] += 200
print(ucty)
```

> **Hint:** `Banka` drží slovník účtů, `prevod(od, kam, castka)` vytvoří `Transakci` a uloží ji; `Ucet` má `zustatek` jako property s validací.


## Metodické poznámky pro výuku OOP

| Téma | Tip pro výklad |
|---|---|
| **Třída vs. instance** | Ukaž `id(obj)` dvou instancí – studenti vidí, že jsou to různé objekty. |
| **`self`** | Vysvětli, že `obj.metoda(x)` je jen zkratka za `Trida.metoda(obj, x)`. |
| **Property** | Přemosť od veřejného atributu → validace v metodě → `@property` (evoluce návrhu). |
| **Dědičnost** | Začni `Zvire`/`Pes`, ale hned ukaž **proč** je kompozice často lepší (příklad 7). |
| **Dunder** | Nech studenty napsat `print(obj)` bez `__str__` (ošklivý výstup) a pak s ním. |
| **ABC** | Ukaž chybu `TypeError: Can't instantiate abstract class` – studenti si to zapamatují. |
| **Kompozice** | Motto: *"Preferuj kompozici před dědičností"* – uveď reálné příklady z frameworků. |

**Doporučené pořadí výuky:**
1 → 2 → 3 → 4 (základy)
→ 5 → 7 (třídní vs. instanční, kompozice)
→ 6 → 8 (dunder, kontejnery)
→ 10 → 11 (dataclass, ABC)
→ 9 → 12 (pokročilé)

**Hodnocení cvičení:** funkčnost 40 %, čistota a validace 30 %, dodržení OOP principů (zapouzdření, polymorfismus) 30 %.

Chceš k těmto cvičením **vzorová řešení** (např. prvních 6 pro kontrolu), nebo připravit **startovací šablony** souborů s prázdnými třídami a docstringy pro studenty?

# Startovací šablony pro OOP cvičení

Níže najdeš **12 startovacích šablon** – prázdné třídy s docstringy, typovými nápovědami a `# TODO` komentáři. Studenti doplní logiku. Každá šablona má i **kostru testů** (asserty), aby studenti hned viděli, co mají splnit.

Doporučuji strukturu projektu:

```
oop_cviceni/
├── cviceni_01_heslo.py
├── cviceni_02_teplota.py
├── ...
├── cviceni_12_banka.py
├── testy/
│   ├── test_01.py
│   ├── test_02.py
│   └── ...
└── README.md

## Cvičení 1 – Základy třídy

**`cviceni_01_heslo.py`**


In [ ]:
"""Cvičení 1: Třída Heslo – základy třídy, __init__, __str__."""


class Heslo:
    """Zapouzdřuje heslo a umí ověřit jeho bezpečnost.

    Atributy:
        _heslo (str): samotné heslo (interní, neveřejné)
    """

    def __init__(self, heslo: str) -> None:
        # TODO: ulož heslo do self._heslo
        ...

    def je_bezpecne(self) -> bool:
        """Vrátí True, pokud má heslo:
        - alespoň 8 znaků
        - obsahuje alespoň jednu číslici
        - obsahuje alespoň jedno velké písmeno
        """
        # TODO: implementuj kontrolu
        ...

    def __str__(self) -> str:
        """Vrátí 'Heslo (skryto)' – nikdy nevypisuj skutečné heslo."""
        # TODO
        ...


In [ ]:
if __name__ == "__main__":
    h = Heslo("Abc12345")
    print(h)                       # Heslo (skryto)
    print(h.je_bezpecne())         # True
    print(Heslo("slabe").je_bezpecne())  # False

**`testy/test_01.py`**

In [ ]:
from cviceni_01_heslo import Heslo

def test_bezpecne():
    assert Heslo("Abc12345").je_bezpecne() is True

def test_kratke():
    assert Heslo("Ab1").je_bezpecne() is False

def test_bez_cislice():
    assert Heslo("Abcdefgh").je_bezpecne() is False

def test_bez_velkeho():
    assert Heslo("abcdefg1").je_bezpecne() is False

def test_str_skryva():
    assert str(Heslo("Tajne123")) == "Heslo (skryto)"

## Cvičení 2 – Property

**`cviceni_02_teplota.py`**


In [11]:
"""Cvičení 2: Třída Teplota – @property, validace, odvozené vlastnosti."""


class Teplota:
    """Teplota s validací a převody na °F a K.

    Rozsah: -273.15 až 1000 °C
    """

    def __init__(self, celsius: float) -> None:
        # TODO: nastav self.celsius (použije se setter)
        ...

    @property
    def celsius(self) -> float:
        # TODO: vrať self._celsius
        ...

    @celsius.setter
    def celsius(self, hodnota: float) -> None:
        # TODO: validuj rozsah, při chybě raise ValueError
        # TODO: ulož do self._celsius
        ...

    @property
    def fahrenheit(self) -> float:
        """Převod: F = C * 9/5 + 32 (read-only)."""
        # TODO
        ...

    @property
    def kelvin(self) -> float:
        """Převod: K = C + 273.15 (read-only)."""
        # TODO
        ...



In [12]:
if __name__ == "__main__":
    t = Teplota(25)
    print(t.fahrenheit)   # 77.0
    print(t.kelvin)       # 298.15
    t.celsius = 100
    print(t.fahrenheit)   # 212.0

None
None
None


In [13]:
import pytest
from cviceni_02_teplota import Teplota

def test_prevod_fahrenheit():
    assert Teplota(0).fahrenheit == 32.0

def test_prevod_kelvin():
    assert Teplota(0).kelvin == 273.15

def test_validace_rozsah():
    with pytest.raises(ValueError):
        Teplota(-300)

def test_setter():
    t = Teplota(0)
    t.celsius = 100
    assert t.fahrenheit == 212.0

ModuleNotFoundError: No module named 'cviceni_02_teplota'

## Cvičení 3 – Dědičnost

**`cviceni_03_zamestnanci.py`**

In [ ]:
"""Cvičení 3: Dědičnost – Zamestnanec, Manazer, Programator."""


class Zamestnanec:
    """Základní zaměstnanec se jménem a základním platem."""

    def __init__(self, jmeno: str, plat: float) -> None:
        # TODO: ulož jmeno a plat
        ...

    def celkova_odmena(self) -> float:
        """Vrátí celkovou odměnu (základ)."""
        # TODO
        ...

    def __str__(self) -> str:
        return f"{type(self).__name__} {self.jmeno}: {self.celkova_odmena():.0f} Kč"


class Manazer(Zamestnanec):
    """Manazer má navíc příplatek."""

    def __init__(self, jmeno: str, plat: float, priplatek: float) -> None:
        # TODO: super().__init__(jmeno, plat), ulož priplatek
        ...

    def celkova_odmena(self) -> float:
        # TODO: plat + priplatek
        ...


class Programator(Zamestnanec):
    """Programator má bonus za projekt."""

    def __init__(self, jmeno: str, plat: float, bonus: float) -> None:
        # TODO
        ...

    def celkova_odmena(self) -> float:
        # TODO: plat + bonus
        ...


def celkove_naklady(zamestnanci: list[Zamestnanec]) -> float:
    """Sečte odměny všech zaměstnanců (polymorfismus)."""
    # TODO
    ...



In [ ]:
if __name__ == "__main__":
    tym = [
        Zamestnanec("Anna", 30000),
        Manazer("Petr", 50000, 10000),
        Programator("Jana", 45000, 5000),
    ]
    for z in tym:
        print(z)
    print(f"Celkem: {celkove_naklady(tym):.0f} Kč")

**`testy/test_03.py`**

In [ ]:
from cviceni_03_zamestnanci import Zamestnanec, Manazer, Programator, celkove_naklady

def test_manazer():
    assert Manazer("P", 50000, 10000).celkova_odmena() == 60000

def test_programator():
    assert Programator("J", 45000, 5000).celkova_odmena() == 50000

def test_polymorfismus():
    tym = [Zamestnanec("A", 30000), Manazer("P", 50000, 10000)]
    assert celkove_naklady(tym) == 90000

## Cvičení 4 – Dunder metody

**`cviceni_04_penize.py`**

In [ ]:
"""Cvičení 4: Třída Penize – dunder metody, formátování."""


class Penize:
    """Částka v dané měně. Podporuje +, -, ==, __str__.

    Operace mezi různými měnami → ValueError.
    """

    def __init__(self, castka: float, mena: str = "Kč") -> None:
        # TODO
        ...

    def __add__(self, other: "Penize") -> "Penize":
        # TODO: zkontroluj měnu, vrať nový Penize
        ...

    def __sub__(self, other: "Penize") -> "Penize":
        # TODO
        ...

    def __eq__(self, other: object) -> bool:
        # TODO: porovnej částku i měnu
        ...

    def __str__(self) -> str:
        """Formát: '1 250,00 Kč' (tisíce oddělené mezerou, desetinná čárka)."""
        # TODO
        ...


In [ ]:
if __name__ == "__main__":
    a = Penize(1000)
    b = Penize(250)
    print(a + b)     # 1 250,00 Kč
    print(a - b)     # 750,00 Kč
    print(a == Penize(1000))  # True
    print(a + Penize(10, "EUR"))  # ValueError

**`testy/test_04.py`**

In [ ]:
import pytest
from cviceni_04_penize import Penize

def test_scitani():
    assert (Penize(1000) + Penize(250)).castka == 1250

def test_odcitani():
    assert (Penize(1000) - Penize(250)).castka == 750

def test_rovnost():
    assert Penize(1000) == Penize(1000)

def test_ruzne_meny():
    with pytest.raises(ValueError):
        Penize(1000) + Penize(10, "EUR")

def test_str_format():
    assert str(Penize(1250)) == "1 250,00 Kč"

## Cvičení 5 – Kompozice

**`cviceni_05_knihovna.py`**


In [ ]:
"""Cvičení 5: Kompozice – Autor a Kniha (HAS-A vztah)."""


class Autor:
    def __init__(self, jmeno: str, rok_narozeni: int) -> None:
        # TODO
        ...

    def __str__(self) -> str:
        return f"{self.jmeno} ({self.rok_narozeni})"


class Kniha:
    """Kniha MÁ autora (kompozice, ne dědičnost)."""

    def __init__(self, nazev: str, autor: Autor, rok_vydani: int) -> None:
        # TODO: self.autor = autor (kompozice!)
        ...

    def stari_pri_vydani(self) -> int:
        """Kolik bylo autorovi let při vydání knihy."""
        # TODO
        ...

    def __str__(self) -> str:
        return f"{self.nazev} – {self.autor.jmeno} ({self.rok_vydani})"


def nejstarsi_kniha(knihy: list[Kniha]) -> Kniha | None:
    """Vrátí knihu s nejmenším rokem vydání (nebo None pro prázdný seznam)."""
    # TODO: použij min() s key
    ...


In [ ]:
if __name__ == "__main__":
    capek = Autor("Karel Čapek", 1890)
    k1 = Kniha("R.U.R.", capek, 1920)
    k2 = Kniha("Bílá nemoc", capek, 1937)
    print(k1)
    print(k1.stari_pri_vydani())   # 30
    print(nejstarsi_kniha([k1, k2]))

**`testy/test_05.py`**

In [ ]:
from cviceni_05_knihovna import Autor, Kniha, nejstarsi_kniha

def test_stari_pri_vydani():
    a = Autor("Karel Čapek", 1890)
    assert Kniha("R.U.R.", a, 1920).stari_pri_vydani() == 30

def test_nejstarsi():
    a = Autor("X", 1900)
    k1 = Kniha("A", a, 1950)
    k2 = Kniha("B", a, 1920)
    assert nejstarsi_kniha([k1, k2]).nazev == "B"

def test_prazdny():
    assert nejstarsi_kniha([]) is None


## Cvičení 6 – Abstraktní třídy

**`cviceni_06_zvirata.py`**


In [ ]:
"""Cvičení 6: ABC – abstraktní Zvire a potomci."""

from abc import ABC, abstractmethod


class Zvire(ABC):
    """Abstraktní zvíře – nelze vytvořit instanci."""

    def __init__(self, jmeno: str) -> None:
        self.jmeno = jmeno

    @abstractmethod
    def zvuk(self) -> str:
        """Každý potomek musí implementovat."""
        ...

    def popis(self) -> str:
        """Vrátí '<Třída> <jméno>: <zvuk>'."""
        # TODO: použij type(self).__name__, self.jmeno, self.zvuk()
        ...


class Pes(Zvire):
    def zvuk(self) -> str:
        # TODO: "Haf!"
        ...


class Kocka(Zvire):
    def zvuk(self) -> str:
        # TODO: "Mňau!"
        ...


class Krava(Zvire):
    def zvuk(self) -> str:
        # TODO: "Búú!"
        ...


def vsechny_zvuky(zvirata: list[Zvire]) -> list[str]:
    """Vrátí seznam zvuků (polymorfismus)."""
    # TODO
    ...


In [ ]:
if __name__ == "__main__":
    zvirata = [Pes("Rex"), Kocka("Micka"), Krava("Bela")]
    for z in zvirata:
        print(z.popis())
    print(vsechny_zvuky(zvirata))   # ['Haf!', 'Mňau!', 'Búú!']

**`testy/test_06.py`**

In [ ]:
import pytest
from cviceni_06_zvirata import Zvire, Pes, Kocka, Krava, vsechny_zvuky

def test_abstraktni():
    with pytest.raises(TypeError):
        Zvire("X")

def test_zvuky():
    zvirata = [Pes("R"), Kocka("M"), Krava("B")]
    assert vsechny_zvuky(zvirata) == ["Haf!", "Mňau!", "Búú!"]

def test_popis():
    assert Pes("Rex").popis() == "Pes Rex: Haf!"

## Cvičení 7 – Kontejner

**`cviceni_07_fronta.py`**


In [ ]:
"""Cvičení 7: Fronta – kontejnerové dunder metody a vlastní výjimka."""

from collections import deque


class PrazdnaFronta(Exception):
    """Vyhozeno při dequeue z prázdné fronty."""
    pass


class Fronta:
    """FIFO fronta s iterací, len a bool."""

    def __init__(self) -> None:
        # TODO: self._data = deque()
        ...

    def enqueue(self, prvek) -> None:
        # TODO
        ...

    def dequeue(self):
        """Vyjmi první prvek. Pokud prázdná → PrazdnaFronta."""
        # TODO
        ...

    def __len__(self) -> int:
        # TODO
        ...

    def __bool__(self) -> bool:
        # TODO: True pokud není prázdná
        ...

    def __iter__(self):
        # TODO: iteruj přes kopii dat (aby se neměnila během iterace)
        ...


In [ ]:
if __name__ == "__main__":
    f = Fronta()
    print(bool(f))          # False
    f.enqueue(1); f.enqueue(2)
    print(len(f))           # 2
    print(f.dequeue())      # 1
    print(list(f))          # [2]
    f.dequeue()
    f.dequeue()             # PrazdnaFronta

**`testy/test_07.py`**

In [14]:
import pytest
from cviceni_07_fronta import Fronta, PrazdnaFronta

def test_zakladni_operace():
    f = Fronta()
    f.enqueue(1); f.enqueue(2)
    assert len(f) == 2
    assert f.dequeue() == 1
    assert bool(f) is True

def test_prazdna():
    with pytest.raises(PrazdnaFronta):
        Fronta().dequeue()

def test_iterace():
    f = Fronta()
    for i in range(3):
        f.enqueue(i)
    assert list(f) == [0, 1, 2]

ModuleNotFoundError: No module named 'cviceni_07_fronta'

## Cvičení 8 – Třídní atributy

**`cviceni_08_student.py`**


In [ ]:
"""Cvičení 8: Třídní atributy – automatické ID a classmethod."""


class Student:
    """Student s automatickým ID a známkami."""

    pocitadlo = 0   # třídní atribut – sdílený

    def __init__(self, jmeno: str, znamky: list[int] | None = None) -> None:
        # TODO: inkrementuj Student.pocitadlo, přiděl self.id
        # TODO: self.jmeno, self.znamky = znamky or []
        ...

    def prumer(self) -> float | None:
        """Průměr známek, nebo None pro prázdný seznam."""
        # TODO
        ...

    @classmethod
    def z_csv(cls, text: str) -> list["Student"]:
        """Parsuje text:
        'Anna;1,2,3\\nPetr;2,2' → [Student('Anna', [1,2,3]), Student('Petr', [2,2])]
        """
        # TODO: rozděl po řádcích a střednících
        ...

    def __str__(self) -> str:
        return f"#{self.id} {self.jmeno}: průměr {self.prumer()}"



In [ ]:
if __name__ == "__main__":
    s1 = Student("Anna", [1, 2, 3])
    s2 = Student("Petr", [2, 2])
    print(s1)                   # #1 Anna: průměr 2.0
    print(s2)                   # #2 Petr: průměr 2.0

    studenti = Student.z_csv("Jana;1,1,2\nKarel;3,4")
    print([str(s) for s in studenti])

**`testy/test_08.py`**

In [ ]:
from cviceni_08_student import Student

def test_auto_id():
    a = Student("A")
    b = Student("B")
    assert b.id == a.id + 1

def test_prumer():
    assert Student("A", [1, 1, 2]).prumer() == 4 / 3

def test_prazdne_znamky():
    assert Student("A").prumer() is None

def test_z_csv():
    studenti = Student.z_csv("Anna;1,2\nPetr;2,2")
    assert len(studenti) == 2
    assert studenti[0].jmeno == "Anna"
    assert studenti[0].znamky == [1, 2]

## Cvičení 9 – Dědičnost od vestavěného typu

**`cviceni_09_slovnik.py`**


In [ ]:
"""Cvičení 9: KlicSlovnik – dědí od dict, ale bezpečnější."""


class KlicSlovnik(dict):
    """Slovník, který:
    - při přístupu na neexistující klíč vrátí None (místo KeyError)
    - klíče typu str převádí na lowercase
    """

    def __getitem__(self, klic):
        # TODO: vrať None, pokud klíč neexistuje
        ...

    def __setitem__(self, klic, hodnota) -> None:
        # TODO: pokud isinstance(klic, str): klic = klic.lower()
        ...

    def __contains__(self, klic) -> bool:
        # TODO: zohledni lowercase
        ...

    def get(self, klic, default=None):
        # TODO: přepiš kvůli konzistenci s __getitem__
        ...



In [ ]:
if __name__ == "__main__":
    d = KlicSlovnik()
    d["Jmeno"] = "Anna"        # uloží se jako "jmeno"
    print(d["jmeno"])          # Anna
    print(d["neexistuje"])     # None (ne KeyError!)
    print("JMENO" in d)        # True
    print(d.get("Jmeno"))      # Anna

**`testy/test_09.py`**

In [ ]:
from cviceni_09_slovnik import KlicSlovnik

def test_lowercase():
    d = KlicSlovnik()
    d["Jmeno"] = "Anna"
    assert d["jmeno"] == "Anna"

def test_none_misto_keyerror():
    assert KlicSlovnik()["cokoliv"] is None

def test_contains():
    d = KlicSlovnik()
    d["Klíč"] = 1
    assert "KLÍČ" in d

def test_get():
    d = KlicSlovnik()
    d["A"] = 1
    assert d.get("a") == 1
    assert d.get("b", 99) == 99

## Cvičení 10 – Observer pattern

**`cviceni_10_observer.py`**


In [ ]:
"""Cvičení 10: Návrhový vzor Observer."""

from typing import Protocol


class Pozorovatel(Protocol):
    """Protokol – cokoliv s metodou oznam()."""
    def oznam(self, zprava: str) -> None: ...


class Novinky:
    """Subject – drží seznam pozorovatelů a rozesílá zprávy."""

    def __init__(self) -> None:
        # TODO: self._pozorovatele = []
        ...

    def prihlas(self, pozorovatel: Pozorovatel) -> None:
        # TODO
        ...

    def odhlas(self, pozorovatel: Pozorovatel) -> None:
        # TODO
        ...

    def posli(self, zprava: str) -> None:
        """Rozešle zprávu všem pozorovatelům."""
        # TODO
        ...


class Uzivatel:
    """Pozorovatel – přijme zprávu a uloží si ji."""

    def __init__(self, jmeno: str) -> None:
        self.jmeno = jmeno
        self.prijate: list[str] = []

    def oznam(self, zprava: str) -> None:
        # TODO: přidej do self.prijate
        ...



In [ ]:
if __name__ == "__main__":
    n = Novinky()
    a = Uzivatel("Anna")
    p = Uzivatel("Petr")
    n.prihlas(a); n.prihlas(p)
    n.posli("Nový článek!")
    print(a.prijate)   # ['Nový článek!']
    n.odhlas(p)
    n.posli("Další")
    print(p.prijate)   # ['Nový článek!'] – Petr už nedostal

**`testy/test_10.py`**

In [ ]:
from cviceni_10_observer import Novinky, Uzivatel

def test_rozesilani():
    n = Novinky()
    a = Uzivatel("A")
    n.prihlas(a)
    n.posli("Zpráva")
    assert a.prijate == ["Zpráva"]

def test_odhlaseni():
    n = Novinky()
    a = Uzivatel("A")
    n.prihlas(a)
    n.odhlas(a)
    n.posli("X")
    assert a.prijate == []

def test_vice_pozorovatelu():
    n = Novinky()
    a, b = Uzivatel("A"), Uzivatel("B")
    n.prihlas(a); n.prihlas(b)
    n.posli("Hi")
    assert a.prijate == b.prijate == ["Hi"]

## Cvičení 11 – E-shop (kombinace)

**`cviceni_11_eshop.py`**

In [ ]:
"""Cvičení 11: E-shop – ABC, kontejner, kompozice, validace."""

from abc import ABC, abstractmethod


class Produkt(ABC):
    def __init__(self, nazev: str, cena_bez_dph: float) -> None:
        if cena_bez_dph < 0:
            raise ValueError("Cena nesmí být záporná")
        self.nazev = nazev
        self.cena_bez_dph = cena_bez_dph

    @property
    @abstractmethod
    def sazba_dph(self) -> float:
        """Každý potomek definuje sazbu."""
        ...

    @property
    def cena_s_dph(self) -> float:
        # TODO: round(cena_bez_dph * (1 + sazba_dph / 100), 2)
        ...

    @abstractmethod
    def popis(self) -> str:
        ...


class Kniha(Produkt):
    @property
    def sazba_dph(self) -> float:
        return 0   # knihy jsou osvobozené
    def popis(self) -> str:
        # TODO: "Kniha: <nazev>"
        ...


class Elektronika(Produkt):
    @property
    def sazba_dph(self) -> float:
        return 21
    def popis(self) -> str:
        # TODO
        ...


class Potravina(Produkt):
    @property
    def sazba_dph(self) -> float:
        return 12
    def popis(self) -> str:
        # TODO
        ...


class Kosik:
    """Kontejner produktů."""

    def __init__(self) -> None:
        # TODO: self._polozky = []
        ...

    def pridej(self, produkt: Produkt) -> None:
        # TODO
        ...

    def __len__(self) -> int:
        # TODO
        ...

    def __iter__(self):
        # TODO
        ...

    def celkem_s_dph(self) -> float:
        # TODO: sum(p.cena_s_dph for p in self._polozky)
        ...


class Objednavka:
    """Objednávka s kupónem."""

    def __init__(self, kosik: Kosik) -> None:
        # TODO
        ...

    def aplikuj_kupon(self, procenta: float) -> None:
        """Uloží slevu (0–100), validuj."""
        # TODO
        ...

    def celkem(self) -> float:
        """Celková cena po slevě."""
        # TODO
        ...


In [ ]:
if __name__ == "__main__":
    k = Kosik()
    k.pridej(Kniha("R.U.R.", 200))
    k.pridej(Elektronika("Sluchátka", 1000))
    k.pridej(Potravina("Chleba", 50))
    print(len(k))               # 3
    print(f"{k.celkem_s_dph():.2f}")   # 200 + 1210 + 56 = 1466.00

    o = Objednavka(k)
    o.aplikuj_kupon(10)
    print(f"{o.celkem():.2f}")  # 1319.40

**`testy/test_11.py`**

In [ ]:
import pytest
from cviceni_11_eshop import Kniha, Elektronika, Potravina, Kosik, Objednavka

def test_dph():
    assert Kniha("X", 100).cena_s_dph == 100
    assert Elektronika("X", 100).cena_s_dph == 121
    assert Potravina("X", 100).cena_s_dph == 112

def test_kosik():
    k = Kosik()
    k.pridej(Kniha("A", 100))
    assert len(k) == 1
    assert k.celkem_s_dph() == 100

def test_kupon():
    k = Kosik()
    k.pridej(Kniha("A", 1000))
    o = Objednavka(k)
    o.aplikuj_kupon(10)
    assert o.celkem() == 900

def test_kupon_validace():
    o = Objednavka(Kosik())
    with pytest.raises(ValueError):
        o.aplikuj_kupon(150)

## Cvičení 12 – Refaktoring do OOP

**`cviceni_12_banka.py`**

In [ ]:
"""Cvičení 12: Refaktoring procedurálního kódu do OOP.

Původní kód:
    ucty = {"Anna": 1000, "Petr": 500}
    ucty["Anna"] -= 200
    ucty["Petr"] += 200
    print(ucty)

Přepiš do tříd Ucet, Transakce, Banka.
"""

from dataclasses import dataclass, field
from datetime import datetime


class Ucet:
    """Účet s validovaným zůstatkem (property)."""

    def __init__(self, majitel: str, zustatek: float = 0) -> None:
        # TODO: self.majitel, self._zustatek (přes setter)
        ...

    @property
    def zustatek(self) -> float:
        # TODO
        ...

    @zustatek.setter
    def zustatek(self, hodnota: float) -> None:
        # TODO: validuj >= 0
        ...


@dataclass
class Transakce:
    """Záznam o převodu."""
    od: str
    kam: str
    castka: float
    cas: datetime = field(default_factory=datetime.now)


class Banka:
    """Drží účty, provádí převody, eviduje transakce."""

    def __init__(self) -> None:
        # TODO: self._ucty: dict[str, Ucet] = {}, self._historie: list[Transakce] = []
        ...

    def zaloz_ucet(self, majitel: str, zustatek: float = 0) -> Ucet:
        # TODO: validuj duplicitu, přidej do _ucty
        ...

    def prevod(self, od: str, kam: str, castka: float) -> None:
        """Převeď částku. Validuj existenci účtů a dostatek prostředků."""
        # TODO: zkontroluj účty, proveď, přidej Transakci do _historie
        ...

    def historie(self) -> list[Transakce]:
        # TODO: vrať kopii historie
        ...


In [ ]:
if __name__ == "__main__":
    b = Banka()
    b.zaloz_ucet("Anna", 1000)
    b.zaloz_ucet("Petr", 500)
    b.prevod("Anna", "Petr", 200)
    print(b._ucty["Anna"].zustatek)   # 800
    print(b._ucty["Petr"].zustatek)   # 700
    for t in b.historie():
        print(f"{t.od} → {t.kam}: {t.castka}")

**`testy/test_12.py`**

In [ ]:
import pytest
from cviceni_12_banka import Banka, Ucet

def test_zalozeni():
    b = Banka()
    b.zaloz_ucet("Anna", 1000)
    assert b._ucty["Anna"].zustatek == 1000

def test_prevod():
    b = Banka()
    b.zaloz_ucet("A", 1000)
    b.zaloz_ucet("B", 0)
    b.prevod("A", "B", 300)
    assert b._ucty["A"].zustatek == 700
    assert b._ucty["B"].zustatek == 300

def test_nedostatek():
    b = Banka()
    b.zaloz_ucet("A", 100)
    b.zaloz_ucet("B", 0)
    with pytest.raises(ValueError):
        b.prevod("A", "B", 500)

def test_historie():
    b = Banka()
    b.zaloz_ucet("A", 100)
    b.zaloz_ucet("B", 0)
    b.prevod("A", "B", 50)
    assert len(b.historie()) == 1

def test_zaporny_zustatek():
    with pytest.raises(ValueError):
        Ucet("X", -100)

In [2]:
class KladnySeznam(list):
    def _zkontroluj(self, hodnota):
        if not isinstance(hodnota, (int, float)) or hodnota <= 0:
            raise ValueError(f"Jen kladná čísla, dostal jsem {hodnota!r}")

    def append(self, hodnota):
        self._zkontroluj(hodnota)
        super().append(hodnota)

    def extend(self, hodnoty):
        for h in hodnoty:
            self._zkontroluj(h)
        super().extend(hodnoty)

    def insert(self, index, hodnota):
        self._zkontroluj(hodnota)
        super().insert(index, hodnota)



In [3]:
s = KladnySeznam([1, 2, 3])
s.append(4)
print(s)                    # [1, 2, 3, 4]
s.append(-1)                # ValueError
s.extend([5, -2])           # ValueError (atomicky – nic se nepřidá)


[1, 2, 3, 4]


ValueError: Jen kladná čísla, dostal jsem -1

# Proč je v metodě super()

# Proč je v metodách `super()`?

Krátká odpověď: **`super()` zavolá původní metodu z nadtřídy (`list`), kterou jsi přepsal.** Bez něj bys jen zkontroloval hodnotu a pak ji **nikam neuložil** – seznam by zůstal prázdný.

---

## Rozbor na `append`

```python
class KladnySeznam(list):

    def append(self, hodnota):
        self._zkontroluj(hodnota)   # 1) moje validace
        super().append(hodnota)     # 2) skutečné uložení do seznamu
```

Když napíšeš:

```python
s = KladnySeznam()
s.append(5)
```

Python hledá metodu `append` **nejdřív ve `KladnySeznam`** → najde tvoji. Ta udělá dvě věci:

1. `self._zkontroluj(5)` – ověří, že 5 je kladné číslo.
2. `super().append(5)` – **deleguje na `list.append`**, který hodnotu skutečně vloží do interního pole.

### Co by se stalo bez `super()`

```python
class KladnySeznam(list):
    def append(self, hodnota):
        self._zkontroluj(hodnota)
        # žádné super().append(hodnota)
```

```python
s = KladnySeznam()
s.append(5)
print(s)        # []  ← prázdné!
print(len(s))   # 0
```

Validace proběhne, ale **data se nikam neuloží**. Tvoje `append` *nahradila* (přepsala) `list.append` a nic ji nenahradilo. To je klasická chyba začátečníků.

---

## Co `super()` technicky dělá

`super()` vrací **proxy objekt**, který hledá metodu v nadtřídě podle **MRO** (Method Resolution Order).

```python
class KladnySeznam(list):
    def append(self, hodnota):
        super().append(hodnota)
        # ^ ekvivalentní zápisu: list.append(self, hodnota)
```

Proč raději `super()` než `list.append(self, ...)`?

| `super().append(x)` | `list.append(self, x)` |
|---|---|
| Funguje i při **vícenásobné dědičnosti** (respektuje MRO) | Pevně ukazuje na `list` |
| Přežije **refaktoring** – změníš rodiče, `super()` jde s tebou | Musíš měnit ručně |
| Standardní idiomatický Python | Funguje, ale křehčí |

### Ukázka MRO

```python
class A:
    def pozdrav(self): print("A")

class B(A):
    def pozdrav(self):
        print("B")
        super().pozdrav()      # → A.pozdrav

class C(A):
    def pozdrav(self):
        print("C")
        super().pozdrav()      # → A.pozdrav

class D(B, C):
    def pozdrav(self):
        print("D")
        super().pozdrav()      # → B.pozdrav (podle MRO: D → B → C → A)

D().pozdrav()
# D
# B
# C
# A
```

`super()` v `D` nešlo rovnou do `A`, ale do **další třídy v MRO** (`B`). To je síla `super()` – každá třída zavolá "svého následníka" a řetěz se složí správně.

---

## Proč je `super()` potřeba i v `extend` a `insert`

Stejný princip – přepisuješ metodu z `list`, takže musíš původní chování **zachovat** a jen ho **obohatit** o validaci:

```python
def extend(self, hodnoty):
    for h in hodnoty:               # 1) zkontroluj VŠECHNO dopředu
        self._zkontroluj(h)
    super().extend(hodnoty)         # 2) teprve pak přidej
```

Poznámka: kontrola **před** `super().extend()` zajišťuje **atomicitu** – když `[5, -2]` obsahuje nevalidní prvek, `extend` vyhodí výjimku dřív, než cokoliv přidá. Kdybys kontroloval až po `super().extend()`, část dat by už byla v seznamu (nekonzistentní stav).

---

## Kdy `super()` **není** potřeba

Když metodu **nepřepisuješ**, ale píšeš úplně novou:

```python
class KladnySeznam(list):
    def secti(self):                # nová metoda – v list neexistuje
        return sum(self)            # žádné super() – není co volat
```

Nebo když metodu přepisuješ **úplně** a chování rodiče nechceš (časté u `__init__` v ABC, kde `super().__init__()` voláš jen kvůli společné inicializaci).

---

## Alternativa: kompozice místo dědičnosti

Pro úplnost – tento příklad jde napsat i **bez dědičnosti**, jen s validací přes vlastní obal:

```python
class KladnySeznam:
    def __init__(self, data=()):
        self._data = []
        for h in data:
            self.append(h)

    def _zkontroluj(self, h):
        if not isinstance(h, (int, float)) or h <= 0:
            raise ValueError(f"Jen kladná čísla, dostal jsem {h!r}")

    def append(self, h):
        self._zkontroluj(h)
        self._data.append(h)

    def __len__(self):  return len(self._data)
    def __iter__(self): return iter(self._data)
    def __getitem__(self, i): return self._data[i]
    def __repr__(self): return f"KladnySeznam({self._data})"
```

**Výhody:** plná kontrola nad všemi cestami vkládání (`+=`, `[0] = x`, `extend`, `insert` …). **Nevýhody:** musíš ručně implementovat každou metodu, kterou chceš podporovat.

U dědičnosti od `list` je **past**: přepíšeš `append`/`extend`/`insert`, ale zapomeneš třeba `__iadd__` (`+=`) nebo `__setitem__` (`s[0] = -1`), které validaci obejdou. To je častý argument pro **kompozici**.

---

## TL;DR

| Otázka | Odpověď |
|---|---|
| **Proč `super()` v `append`?** | Aby se hodnota po validaci **skutečně uložila** do `list`. |
| **Co se stane bez něj?** | Metoda jen zkontroluje a nic neuloží – seznam zůstane prázdný. |
| **Co `super()` dělá?** | Deleguje na stejnojmennou metodu nadtřídy podle MRO. |
| **Proč ne `list.append(self, x)`?** | `super()` je robustnější při vícenásobné dědičnosti a refaktoringu. |
| **Kdy ho vynechat?** | Když píšeš **novou** metodu, nebo úmyslně nechceš chování rodiče. |